In [ ]:
## 15	7	Feature	[24.649744, 59.414041]	Point	URBAN	Õismäe	<p>&Otilde;ism&auml;e seirejaam paikneb Tallin...	images/stations/oismae.jpg	S05	[1, 3, 4, 6, 21, 23]



## 24.6 ja 59.4 koordinaadid lähevad

## lat 59.41 
## lon 24.65

In [ ]:
## impordid



In [ ]:
## lat 59.41, lon 24.65 timezone +0, forecast days 5, past days 1 month
## pm10, pm2.5, no2, so2, o3
## domain european (11km)
## timeformat iso8601


In [ ]:
## openmeteo lehelt - installid

# !pip install openmeteo-requests
# !pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [31]:
## openmeteo lehelt kood

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": 59.41,
	"longitude": 24.65,
	"hourly": ["pm10", "pm2_5", "ozone", "nitrogen_dioxide", "sulphur_dioxide"],
	"past_days": 31,
	"domains": "cams_europe",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_pm10 = hourly.Variables(0).ValuesAsNumpy()
hourly_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
hourly_ozone = hourly.Variables(2).ValuesAsNumpy()
hourly_nitrogen_dioxide = hourly.Variables(3).ValuesAsNumpy()
hourly_sulphur_dioxide = hourly.Variables(4).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["pm10"] = hourly_pm10
hourly_data["pm2_5"] = hourly_pm2_5
hourly_data["ozone"] = hourly_ozone
hourly_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
hourly_data["sulphur_dioxide"] = hourly_sulphur_dioxide
hourly_data["coordinate_lat"] = params["latitude"]
hourly_data["coordinate_lon"] = params["longitude"]

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 59.400001525878906°N 24.700000762939453°E
Elevation: 12.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  pm10  pm2_5  ozone  nitrogen_dioxide  \
0   2026-04-29 00:00:00+00:00   3.8    2.5   85.0               2.4   
1   2026-04-29 01:00:00+00:00   3.8    2.4   84.0               2.5   
2   2026-04-29 02:00:00+00:00   3.7    2.2   83.0               2.4   
3   2026-04-29 03:00:00+00:00   3.7    2.2   81.0               2.4   
4   2026-04-29 04:00:00+00:00   3.7    2.3   80.0               2.6   
..                        ...   ...    ...    ...               ...   
859 2026-06-03 19:00:00+00:00   NaN    NaN    NaN               NaN   
860 2026-06-03 20:00:00+00:00   NaN    NaN    NaN               NaN   
861 2026-06-03 21:00:00+00:00   NaN    NaN    NaN               NaN   
862 2026-06-03 22:00:00+00:00   NaN    NaN    NaN               NaN   
863 2026-06-03 23:00:00+00:00   NaN    NaN    NaN               NaN   

     sulphur_dioxide  c

In [3]:
type(response)

openmeteo_sdk.WeatherApiResponse.WeatherApiResponse

In [32]:
for rida in hourly_dataframe.iterrows():
    #print(rida)
    
    print("rea nr on: ", rida[0])
    
    print("rea sisu on: ", rida[1])
    # print("rea date on: ", rida[1]["date"])
    # print("rea date on: ", rida[1][0])
    # print("rea pm10 on: ", rida[1]["pm10"])
    # print("rea pm 10 on: ", rida[1][1])
    # # print("rea pm2_5 on: ", rida[1]["pm2_5"])
    # break

rea nr on:  0
rea sisu on:  date                2026-04-29 00:00:00+00:00
pm10                                      3.8
pm2_5                                     2.5
ozone                                    85.0
nitrogen_dioxide                          2.4
sulphur_dioxide                           0.7
coordinate_lat                          59.41
coordinate_lon                          24.65
Name: 0, dtype: object
rea nr on:  1
rea sisu on:  date                2026-04-29 01:00:00+00:00
pm10                                      3.8
pm2_5                                     2.4
ozone                                    84.0
nitrogen_dioxide                          2.5
sulphur_dioxide                           0.7
coordinate_lat                          59.41
coordinate_lon                          24.65
Name: 1, dtype: object
rea nr on:  2
rea sisu on:  date                2026-04-29 02:00:00+00:00
pm10                                      3.7
pm2_5                                     

In [8]:
hourly_dataframe.columns.tolist()

['date', 'pm10', 'pm2_5', 'ozone', 'nitrogen_dioxide', 'sulphur_dioxide']

In [9]:
hourly_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864 entries, 0 to 863
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   date              864 non-null    datetime64[ns, UTC]
 1   pm10              841 non-null    float32            
 2   pm2_5             841 non-null    float32            
 3   ozone             841 non-null    float32            
 4   nitrogen_dioxide  841 non-null    float32            
 5   sulphur_dioxide   841 non-null    float32            
dtypes: datetime64[ns, UTC](1), float32(5)
memory usage: 23.8 KB


In [11]:
hourly_dataframe.head(50)

,date,pm10,pm2_5,ozone,nitrogen_dioxide,sulphur_dioxide
0,2026-04-29 00:00:00+00:00,3.8,2.5,85.0,2.4,0.7
1,2026-04-29 01:00:00+00:00,3.8,2.4,84.0,2.5,0.7
2,2026-04-29 02:00:00+00:00,3.7,2.2,83.0,2.4,0.6
3,2026-04-29 03:00:00+00:00,3.7,2.2,81.0,2.4,0.6
4,2026-04-29 04:00:00+00:00,3.7,2.3,80.0,2.6,0.6
5,2026-04-29 05:00:00+00:00,3.6,2.5,80.0,2.6,0.5
6,2026-04-29 06:00:00+00:00,3.5,2.6,82.0,2.4,0.6
7,2026-04-29 07:00:00+00:00,3.6,2.5,82.0,1.9,0.6
8,2026-04-29 08:00:00+00:00,3.6,2.5,84.0,1.6,0.6
9,2026-04-29 09:00:00+00:00,3.5,2.5,86.0,1.2,0.6
